# report11 — 검출기 교정 ②: 저속 표적·분해능

> ### ❓ 이 리포트가 답하는 질문
> **느린 드론과 신호의 분해능 한계는 검출에 어떤 문턱을 만드나?**

### ⚡ 결론부터 (TL;DR)

1. **호버 드론은 원리적으로 어렵다.** 직접파 제거(ECA)는 정지 성분을 지우는데, 초속 약 **0.29~0.88 m/s** 보다 느린 표적은 도플러가 거의 0 이라 표적까지 함께 지워진다.
2. **분해능 눈금은 맞다.** 각 신호의 거리 분해능은 이론 ΔR_b=c/B 대비 **0.89~0.94배**(직사각 창 sinc 의 0.886 근처)로, 교과서와 일치한다.
3. **밝기·거리·잡음 계산도 맞다.** 링크버짓을 세 가지 독립 방법으로 계산해 최대 편차 **3e-14 dB**, 잡음바닥은 이론과 **1e-15 dB** 로 일치한다.
4. **그래서 탐지가 된다.** SBR 로 계산한 밝기(σ)를 넣으면 5대 드론×3신호 모두 SCR **14.8~55.3 dB** 로 잡히고(Pd=1.0), 검출은 SCR **5~15 dB** 구간에서 전이한다.

### 🗺️ 어디부터 읽나

| 절 | 무엇을 |  |
|---|---|---|
| §1 | ECA 저속 맹점 — 블라인드 속도 | 호버 드론이 왜 어려운가 |
| §2 | 모호함수 — 거리·속도 분해능 | 신호가 표적을 얼마나 또렷이 가르나 |
| §3 | 링크버짓 — 밝기·거리·잡음 → SNR | 계산이 이론과 맞나 |
| §4 | 검출확률 곡선 — 디텍션이 실제로 작동 | σ 를 넣으면 잡힌다는 증거 |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| ECA 노치·블라인드 속도(운용 적분시간) | outputs/verify_eca.json | 측정 (RT 클러터 + ECA 사영) |
| 모호함수·거리 분해능 측정/이론 | outputs/verify_ambiguity.json | 측정 (기준신호 자기모호함수) |
| 레이더방정식·처리이득·잡음바닥·SCR·Pd | outputs/verify_linkbudget.json · outputs/verify_cfar.json(roc_NR100) | 측정 (σ 는 SBR, 3GPP TS / ITU-R P.2040 재질) |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `radar-dsp` | 레이더 신호처리 (`src/passive_process.py`) — ECA(직접파 제거) · 거리-도플러 · CA-CFAR | 🔴 **별도** (numpy, CPU). **Sionna 에 레이더 DSP 는 없다** |
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `sionna-phy` | Sionna PHY (`ofdm`/`nr`/`channel`) — OFDM 변복조 · 3GPP 뉴머롤로지 · RT 경로를 신호에 적용 | 🟢 **Sionna 내부** (PyTorch 백엔드, GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: verify_* 스크립트 각 GPU 1장 수 분. 그림은 이미 산출된 것을 재사용.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python benchmark/verify_eca.py
~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
~/.venvs/py312/bin/python benchmark/verify_linkbudget.py
~/.venvs/py312/bin/python src/make_notebook11.py     # report11.ipynb 재생성
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/verify_eca.json` | ECA 노치·블라인드 속도 |
| `outputs/verify_ambiguity.json` | 모호함수·거리/속도 분해능 |
| `outputs/verify_linkbudget.json` | 레이더방정식·잡음·SCR·Pd |
| `outputs/figures/verify_eca.png` | ECA 저속 맹점 그림 (재사용) |
| `outputs/figures/verify_ambiguity_af.png` | 모호함수 거리-도플러 지도 (재사용) |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **블라인드 속도는 적분시간(CPI)에 달렸다.** 여기 값은 운용 적분시간 기준이다. 더 오래 적분하면 노치가 좁아져 더 느린 표적까지 보이지만 관측이 그만큼 느려진다(맞바꿈).
- **분해능은 신호가 상시 쓰는 기준신호의 대역폭이 정한다.** 5G 의 상시 신호(SSB)는 대역이 7.2 MHz 뿐이라 거리 분해능이 39 m 로 거칠다.
- **여기서 다루는 것은 탐지(거리+속도 셀에서 '있다/없다')까지다.** 표적의 위치·궤적을 잇는 추적은 이 리포트의 범위 밖이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| report10 (앞) | 오경보율 — 검출기가 '헛것'을 얼마나 자주 보나(문턱 눈금) |
| (끝) | 이 리포트가 마지막이다 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **도플러 f_d** | 표적이 움직여 생기는 수신 주파수의 이동. 정지물(클러터)과 움직이는 표적을 가르는 축. 느린 표적일수록 f_d 가 0 에 가까워 클러터와 구별이 어렵다 |
| **ECA (직접파 제거)** | 송신기에서 수신기로 곧장 새어 든 강한 직접파를, 지연된 기준신호가 만드는 부분공간에 사영해 빼는 전처리. **정지 성분(도플러 0)을 통째로 지운다** — 이 성질이 곧 저속 맹점의 뿌리다 |
| **블라인드 속도** | ECA 노치에 먹혀 되돌아온 신호가 3 dB 이상 깎이는 최소 시선속도. 이보다 느리면 표적이 **원리적으로** 잘 안 보인다 |
| **모호함수 / CAF** | 교차모호함수(Cross-Ambiguity Function). 기준신호와 수신신호를 **거리(지연)와 도플러로 동시에** 상관시켜 만든 2차원 지도. 봉우리 폭이 분해능, 곁봉우리가 가짜표적 |
| **거리 분해능 ΔR_b** | 두 표적을 거리축에서 갈라 볼 수 있는 최소 간격. 바이스태틱에서 ΔR_b = c/B (B = 기준신호 대역폭). 대역이 넓을수록 촘촘히 가른다 |
| **sinc / 0.886** | 직사각 창의 스펙트럼 모양(sin πx / πx). 그 −3 dB 폭은 이론값의 **0.886배**다. 측정/이론 비가 이 값 근처면 분해능 눈금이 맞은 것 |
| **링크버짓** | 표적 밝기(σ)·거리·잡음을 곱하고 나눠 되돌아온 신호대잡음비(SNR)를 예측하는 계산. = 레이더 방정식 |
| **RCS (σ)** | 레이더 되비침 밝기. σ 는 그 밝기를 넓이 단위(m²)로 적은 값. 표적이 밝아야(σ 가 커야) 잡힌다. Sionna 기본 solver 는 이 σ 를 못 내므로 **SBR** 로 따로 계산해 넣는다 |
| **SBR** | 표적을 광선으로 조준해 맞고, 맞은 면이 레이더로 되쏘는 양을 물리광학(PO)으로 계산·합산해 σ 를 구하는 방법 |
| **SCR** | 신호대클러터비(Signal-to-Clutter Ratio). 거리-도플러 지도에서 표적 봉우리 ÷ 주변 바닥. 검출 난이도의 실측 지표 |
| **Pd (검출확률)** | 표적이 있을 때 검출기가 실제로 발화할 확률. SCR 이 올라갈수록 0 에서 1 로 전이한다 |
| **CFAR** | 주변 잡음을 보고 문턱을 스스로 정하는 검출기(Constant False Alarm Rate) |
| **CPI / 적분시간** | 한 번 관측하며 신호를 모아 쌓는 시간(Coherent Processing Interval). 길수록 도플러를 촘촘히 보지만 관측이 느려진다 |
| **semi-anechoic** | 벽·천장은 전파를 흡수하고 바닥만 반사하는 반무향 챔버 — 우리 실험 환경 |

</details>

---


## 🔰 5분이면 이해하는 이 리포트

**청소기 비유부터.** 패시브 레이더는 방 안에 이미 켜져 있는 WiFi·LTE·5G 신호를 조명 삼아 드론을 비춘다. 문제는 송신기에서 수신기로 **곧장 새어 드는 직접파**가 표적 메아리보다 수십만 배 세다는 것. 이 직접파를 지우는 청소기가 **ECA(직접파 제거)** 다. 청소기는 '움직이지 않는 먼지'(도플러 0, 정지 성분)를 빨아들이도록 만들어졌다.

**그런데 청소기 옆에 아주 느린 벌레가 기어가면?** 거의 안 움직이니 청소기는 그걸 먼지로 착각해 **같이 빨아 버린다.** 제자리에 떠 있는 호버 드론이 바로 그 느린 벌레다. 초속 약 **0.29~0.88 m/s** 보다 느리면 도플러가 거의 0 이라 직접파와 구별이 안 돼 표적이 함께 지워진다. 이게 **블라인드(맹점) 속도** — 원리적인 한계다(§1).

**다음은 '눈의 해상도'.** 신호가 두 표적을 얼마나 또렷이 갈라 보는지는 **모호함수**라는 지도로 잰다 — 봉우리가 뾰족할수록 거리·속도를 잘 가른다. 봉우리 폭은 신호 대역폭이 정하는데, 측정해 보니 교과서 값의 **0.89~0.94배**(직사각 창의 자연스러운 0.886배)로 딱 맞는다(§2).

**그리고 '밝기 계산'.** 표적이 얼마나 밝게 되돌아오는지(레이더 방정식)를 세 가지 독립 방식으로 계산해 서로 소수점 열 자리까지 같음을 확인한다(§3). 마지막으로, 그 밝기(SBR 로 계산한 σ)를 넣으면 **드론이 실제로 잡힌다** — 검출확률이 SCR 5~15 dB 구간에서 0 에서 1 로 올라서고, 우리 드론들은 그 위에 있어 모두 잡힌다(§4). 즉 **표적을 밝게 만드는 σ 를 SBR 로 넣으면 탐지가 된다.**

> **앞 리포트(report10)** 에서는 검출기가 '헛것'을 얼마나 자주 보는지(오경보율) 문턱 눈금을 맞췄다. 여기서는 반대로 **진짜 표적을 얼마나 놓치고, 얼마나 또렷이 보는지**를 다룬다.

---
# §1. 직접파 제거의 저속 맹점 — 블라인드 속도

> 🔍 **여기서 하는 일:** 정지 클러터를 지우는 ECA 가 **느린 표적을 얼마나 함께 지우는지** 재고, 그 아래로는 못 보는 최소 시선속도를 구한다.

**직관.** ECA 는 도플러가 0 인 성분(움직이지 않는 벽·바닥·직접파)을 지우도록 설계돼 있다. 그런데 표적이 느리면 그 표적의 도플러도 0 에 가까워진다. 청소기가 느린 벌레를 먼지로 착각하듯, ECA 는 **느린 표적을 정지 클러터로 착각해 함께 지운다.** 지워지는 정도는 표적의 도플러가 '도플러 한 칸(=1/적분시간)'에서 얼마나 떨어져 있느냐로 정해진다.

**근거 — 노치의 모양.** ECA 가 표적을 깎는 양은 이론적으로 $1-\mathrm{sinc}^2(f_d\,T)$ 라는 골짜기(노치) 모양을 따른다(T = 적분시간). 측정한 손실 곡선은 이 이론과 최대 **0.002 dB** 밖에 차이 나지 않는다 — 노치 폭은 정확히 **도플러 한 칸**이다. 아래 표는 5G 신호에서 표적 도플러가 도플러 한 칸의 몇 배일 때 얼마나 깎이는지다(이론과 나란히).

| 표적 도플러 ÷ 한 칸 | 측정 에너지 손실 | 이론 $1-\mathrm{sinc}^2$ |
|---|---|---|
| 0.3 | -5.80 dB | -5.80 dB |

**블라인드 속도.** 되돌아온 에코 에너지가 −3 dB(절반)로 깎이는 지점을 시선속도로 환산하면, 운용 적분시간에서 이렇게 나온다.

| 신호 | 적분시간 | 도플러 한 칸 | 블라인드 속도(−3 dB) |
|---|---|---|---|
| 5G NR 100MHz | 24 ms | 41.7 Hz | **0.88 m/s** |
| WiFi 80MHz | 48 ms | 20.8 Hz | **0.29 m/s** |
| LTE 20MHz | 48 ms | 20.8 Hz | **0.83 m/s** |

즉 초속 **0.29~0.88 m/s** 보다 느린 표적은 −3 dB 이상 깎여 사실상 놓친다. **제자리에 떠 있는(호버) 드론은 시선속도가 0 에 가까워 원리적으로 이 골짜기 안에 갇힌다.** 적분시간을 96 ms 로 더 늘리면 노치가 좁아져 **0.15 m/s** 까지 내려가지만, 그만큼 한 번 보는 데 오래 걸린다(관측을 느리게 만드는 맞바꿈).

아래 그림 (c) 는 모든 신호·적분시간의 손실 곡선이 하나의 $1-\mathrm{sinc}^2$ 곡선으로 겹침을, (d) 는 신호별 저속 사각지대(이 속도 아래로는 표적을 먹는다)를 보여 준다.

![ECA blind-speed notch and slow-drone blind zone](outputs/figures/verify_eca.png)

---
# §2. 모호함수 — 신호가 표적을 얼마나 또렷이 가르나

> 🔍 **여기서 하는 일:** 각 신호가 두 표적을 **거리·속도로 얼마나 잘 갈라 보는지**를 모호함수 지도로 재고, 거리 분해능이 교과서 값과 맞는지 확인한다.

**직관.** 손전등을 벽에 비추면 빛 얼룩이 생긴다. 얼룩이 작고 또렷할수록 벽의 두 점을 갈라 볼 수 있다. 레이더에서 이 '빛 얼룩'에 해당하는 게 **모호함수** — 기준신호를 거리(지연)와 속도(도플러)로 동시에 훑어 만든 2차원 지도다. 한가운데 봉우리가 뾰족할수록 두 표적을 잘 가르고(분해능), 봉우리 옆에 솟은 곁봉우리는 없는 표적을 있는 것처럼 보이게 하는 함정(가짜표적)이다.

**근거 — 거리 분해능.** 바이스태틱 거리 분해능의 이론값은 $\Delta R_b = c / B$ (B = 기준신호 대역폭). 직사각 창이면 실제 −3 dB 봉우리 폭은 이론의 **0.886배**(sinc 폭)가 나와야 정상이다. 측정 결과:

| 신호(기준) | 기준 대역폭 | 측정 ΔR_b | 이론 c/B | 측정/이론 | 곁봉우리(챔버) |
|---|---|---|---|---|---|
| WiFi 80 MHz · VHT-LTF | 76.6 MHz | 3.50 m | 3.92 m | **0.895** | -23.4 dB |
| LTE 20 MHz · CRS | 18.0 MHz | 15.33 m | 16.67 m | **0.920** | -15.0 dB |
| 5G NR 100 MHz (상시 SSB) · SSB | 7.2 MHz | 39.21 m | 41.64 m | **0.942** | -18.3 dB |
| 5G NR 100 MHz (측위 PRS) · NR-PRS | 98.3 MHz | 2.77 m | 3.05 m | **0.910** | -15.7 dB |

측정/이론 비가 전부 **0.89~0.94**(sinc 의 0.886 근처)로 거리 눈금이 맞았다. **대역이 넓을수록 촘촘히 가른다** — 5G 의 측위용 신호(NR-PRS, 98 MHz)는 2.8 m 까지 가르지만, 5G 가 **상시** 내보내는 신호(SSB)는 대역이 7.2 MHz 뿐이라 39 m 로 거칠다. 즉 5G 로 상시 탐지하려면 거리 분해능을 크게 손해 본다.

**곁봉우리(가짜표적)는 챔버 밖에 있다.** OFDM 신호의 반복 구조는 곁봉우리를 만들지만, 그 봉우리들은 수백 m~km 거리에 찍혀 우리 챔버 관측창(바이스태틱 거리 ±60 m 안, 도플러 ±260 Hz) 밖이다. 창 안에서 곁봉우리는 표적보다 -23 dB 이상 낮아 무해하다.

**이 모호함수는 실제 검출에 쓰는 거리-도플러 지도와 같은 것이다** — 둘을 맞대 보면 최대 **0.14 dB** 밖에 차이 나지 않는다. 아래 그림은 신호별 모호함수 지도(위)와 거리축 단면(아래)이다.

![Ambiguity function |chi(tau,fd)| per waveform](outputs/figures/verify_ambiguity_af.png)

![Zero-Doppler range cut — mainlobe width = range resolution](outputs/figures/verify_ambiguity_range.png)

---
# §3. 링크버짓 — 밝기·거리·잡음이 신호대잡음비를 만든다

> 🔍 **여기서 하는 일:** 표적이 얼마나 밝게 되돌아오는지(레이더 방정식)를 **독립적인 세 방법**으로 계산해 서로, 그리고 이론과 맞는지 확인한다.

**직관.** 밤에 손전등으로 멀리 있는 표지판을 비춘다고 하자. 눈에 들어오는 밝기는 (표지판이 얼마나 잘 반사하나) × (거리가 멀수록 어두워짐) ÷ (주변이 얼마나 밝아 방해되나)로 정해진다. 레이더도 똑같다: **표적 밝기(σ) × 거리 감쇠 ÷ 잡음** = 되돌아온 신호대잡음비(SNR). 이걸 계산하는 게 **링크버짓**이다.

여기서 표적 밝기 σ 는 Sionna 의 광선추적이 주지 못한다(기본 solver 에 산란적분 단계가 없어 표적 밝기를 못 낸다). 그래서 **SBR** — 표적을 광선으로 조준해 맞고 그 면이 되쏘는 양을 물리광학으로 합산 — 로 σ 를 따로 계산해 넣는다.

**근거 — 세 계산이 일치한다.** 같은 SNR 을 (a) 닫힌형 공식, (b) 단계별 사슬 계산, (c) dB 산술 세 방법으로 구했더니 서로 최대 **3e-14 dB** 밖에 차이 나지 않는다(사실상 완전 일치). 신호를 모아 쌓는 **처리이득**도 이론 대비 **0.096 dB**, **잡음바닥**은 대역폭 보정을 넣으면 이론과 **1e-15 dB** 로 맞는다. 계산 사슬이 통째로 검증됐다.

| 신호 | 처리이득(이론) | 처리이득(측정) | 잡음바닥 오차 |
|---|---|---|---|
| WiFi 80MHz | 1.74 dB | 1.77 dB | -1e-15 dB |
| LTE 20MHz | 15.88 dB | 15.79 dB | +0e+00 dB |
| 5G 100MHz | 19.21 dB | 19.20 dB | +0e+00 dB |

---
# §4. 검출확률 곡선 — 디텍션이 실제로 작동한다  ⭐

> 🔍 **여기서 하는 일:** SBR 로 계산한 표적 밝기(σ)를 링크버짓에 넣어 얻은 SCR 로 실제 검출기를 돌려, **표적이 정말 잡히는지**를 검출확률(Pd)로 확인한다.

**직관.** 앞의 §1~§3 은 '한계'와 '눈금'을 다뤘다. 이제 핵심 질문: **그래서 잡히긴 하나?** 표적 봉우리가 주변 바닥보다 충분히 높으면(SCR 이 크면) 검출기가 발화한다. SCR 이 낮으면 놓치고, 어느 문턱을 넘으면 확실히 잡는다. 그 전이를 검출확률 곡선으로 본다.

**근거 — 우리 드론들은 모두 잡힌다.** SBR 로 σ 를 계산해 넣으니 5 대 드론 × 3 신호 = 15 조합의 측정 SCR 이 **14.8~55.3 dB** 로 나오고, **전부 Pd = 1.0(모두 검출)** 이다. 링크버짓으로 예측한 SCR 과 실제 지도에서 잰 SCR 의 차이는 평균 **-1.9 dB**(±1.3) 로, 이 차이는 오류가 아니라 창 가중 등 **예측 가능한 처리 손실**이다.

| 드론 | 표적 밝기 σ | 예측 SCR | 측정 SCR | 검출 |
|---|---|---|---|---|
| mini5pro | -27.5 dBsm | 15.7 dB | 14.8 dB | ✅ Pd=1.0 |
| mavic4pro | -16.0 dBsm | 27.1 dB | 26.0 dB | ✅ Pd=1.0 |
| matrice4e | -21.5 dBsm | 21.7 dB | 20.7 dB | ✅ Pd=1.0 |
| s1000plus | -14.2 dBsm | 29.0 dB | 27.9 dB | ✅ Pd=1.0 |
| phantom4 | -19.9 dBsm | 23.3 dB | 22.3 dB | ✅ Pd=1.0 |

*(WiFi 조명 예시 — LTE·5G 도 같은 경향. 작은 드론일수록 σ 가 어둡지만 그래도 문턱 위에 있다.)*

**검출은 SCR 5~15 dB 구간에서 전이한다.** SCR 을 바꿔 가며 검출확률을 재면(5G 조명, 운용 오경보율 $10^{-4}$ 기준) 아래처럼 0 에서 1 로 올라선다.

| SCR | 검출확률 Pd |
|---|---|
| +3.4 dB | 0.001 |
| +3.5 dB | 0.002 |
| +3.8 dB | 0.000 |
| +5.1 dB | 0.020 |
| +9.2 dB | 0.447 |
| +14.8 dB | 1.000 |

SCR 5 dB 근처에서는 거의 못 잡다가 15 dB 에 이르면 확실히 잡는다. **우리 드론들의 운용 SCR 은 모두 15 dB 이상**으로 이 전이 구간의 위쪽에 있으니 확실히 검출된다.

> 🔑 **이 절이 이 리포트의 결론이다.** 표적을 밝게 만드는 σ 는 탐지에 반드시 필요하고(밝아야 봉우리가 선다), 그 σ 를 Sionna 광선추적으로는 못 내므로 **SBR 로 계산해 넣는다.** 그렇게 넣으면 **실제로 탐지가 된다** — 위 표가 그 증거다.

---
## 맺음

검출에는 두 종류의 문턱이 있다. **속도 쪽 문턱**: 직접파 제거(ECA)는 정지 성분을 지우므로 초속 약 0.29~0.88 m/s 보다 느린 표적, 특히 호버 드론을 원리적으로 함께 지운다. **분해능 쪽 문턱**: 두 표적을 가르는 거리 해상도는 상시 신호의 대역폭이 정하며, 측정치는 교과서 값의 0.89~0.94배로 정확하다. 그 사이에서, 밝기·거리·잡음을 잇는 링크버짓은 이론과 소수점까지 맞고, **SBR 로 계산한 표적 밝기(σ)를 넣으면 드론이 실제로 검출된다**(SCR 15~55 dB, Pd=1.0).

**다음 단계로 실증실험(USRP X410) 및 추후 트래킹(거리+속도)까지 확장 예정.**